# Feature Engineering — Land Use Segmentation

Extract elevation, slope, NDVI, and land cover features from rasters and shapefiles
for clustering-based land-use segmentation.

In [ ]:
import pandas as pd
import numpy as np
import rasterio
import geopandas as gpd
from scipy.ndimage import sobel

## Feature Engineering Steps

1. Compute slope and aspect from DEM
2. Extract NDVI from multispectral imagery
3. Sample land cover class at each grid point
4. Compute terrain roughness index
5. Assemble pixel-level feature matrix

In [ ]:
# Compute slope from DEM
with rasterio.open('../data/raw/dem.tif') as src:
    elev = src.read(1).astype(float)
    transform = src.transform

dx = sobel(elev, axis=1)
dy = sobel(elev, axis=0)
slope = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))
aspect = np.degrees(np.arctan2(-dy, dx))
print(f'Slope range: {slope.min():.1f} to {slope.max():.1f} degrees')

In [ ]:
# Compute NDVI
with rasterio.open('../data/raw/multispectral.tif') as src:
    red = src.read(3).astype(float)
    nir = src.read(4).astype(float)

ndvi = (nir - red) / (nir + red + 1e-8)

# Build feature matrix (flattened pixels)
features = pd.DataFrame({
    'elevation': elev.flatten(),
    'slope': slope.flatten(),
    'aspect': aspect.flatten(),
    'ndvi': ndvi.flatten(),
})
features = features.dropna()
features.to_parquet('../data/processed/pixel_features.parquet', index=False)
print(f'Feature matrix: {features.shape}')